# 04 — Cite evidence and abstain with a policy

**Level:** Beginner · **Estimated time:** 80–100 minutes · **Scenario:** Harborline Support

You will model citations and claims as structured data, compare abstention policies, audit provenance separately from presentation, enforce an access boundary, and test answerable, ambiguous, unsupported, and intentionally invalid responses.


## How to use this notebook

Work in this order: read the concept, run the deterministic code, change **one** variable, inspect the trace, and write down what changed. The model API is deliberately absent: the learning objective is to understand the evidence system that an LLM would depend on.

**Scenario.** You are building a small, internal assistant for Harborline, a fictional SaaS company. Support needs trustworthy answers about customer communication and production escalation. The corpus is intentionally tiny so every result can be inspected.


## 1. Grounding is a contract, not a citation style

An answer can contain a link and still be unsupported. A trustworthy system preserves evidence identity through retrieval, context construction, generation, validation, and rendering.

```text
question → ranked evidence → policy decision
                    │             ├─ answer: claim + structured citations
                    │             └─ abstain: reason + next safe verification
                    ↓
              provenance audit → presentation
```

The key separation is **data vs. display**. A `Citation` stores a stable chunk ID, source, and retrieval score. Markdown is only a final rendering choice.


In [ ]:
from pathlib import Path
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/beginner/04-citations-abstention/lab.py')).items() if not name.startswith('_')})
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/beginner/02-first-local-rag/lab.py')).items() if not name.startswith('_')})

ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path('../..')
chunks = load_chunks(ROOT / 'examples/data/beginner-docs')
print('Corpus chunks:', len(chunks))


## 2. A citation does and does not prove things

A citation proves that the system chose a known evidence object. It does **not** independently prove that:

- the passage supports every sentence in the response;
- the source is current;
- the user is authorized to see it;
- the retrieval score is calibrated confidence.

Those properties need separate checks. This notebook implements provenance invariants and an abstention policy; later lessons add permission filters and richer evaluation.


In [ ]:
question = 'What should support do for a confirmed payment incident?'
result = answer_with_citations(question, chunks, top_k=3, min_score=0.20)
print(result)
print('\nRendered for a user:\n')
print(render_markdown(result))


## 3. Audit the evidence object

The audit runs before display logic. It verifies that every cited chunk ID is in the known corpus and reports source diversity and score margin. For a production system, add document version, location, content hash, user authorization, and claim-to-evidence mappings.


In [ ]:
audit = audit_answer(result, chunks)
audit


## 4. Abstention has more than one reason

The assistant can safely decline for different reasons:

| Reason | Meaning | Safe next step |
| --- | --- | --- |
| `insufficient-evidence` | no candidate crossed the evidence threshold | request a source or route to a human |
| `ambiguous-evidence` | top candidates are too close under the policy | retrieve more precisely or clarify the question |
| `insufficient-source-diversity` | a policy requires corroboration but only one source is available | locate a second authoritative source |

The right policy depends on the risk of an unsupported answer. Do not silently convert an abstention into generic model knowledge.


In [ ]:
unsupported = answer_with_citations('What is the capital of France?', chunks)
ambiguous = answer_with_citations(
    'What must an answer distinguish?',
    chunks,
    policy=AbstentionPolicy(min_score=0.10, min_margin=0.90),
)

for label, response in {'unsupported': unsupported, 'ambiguous': ambiguous}.items():
    print(label, '→', response.abstained, response.reason)
    print(render_markdown(response), '\n')


## 5. Thresholds are evaluated, not guessed

Lower thresholds reduce abstentions but may admit weak evidence. Higher thresholds can block useful answers. A margin can flag close competing candidates, but it is still a retrieval heuristic rather than a truth signal.

Test your policy on a balanced set:

- answerable questions with known evidence;
- out-of-corpus questions that must abstain;
- ambiguous questions that need clarification;
- queries using paraphrases, identifiers, and partial wording.


In [ ]:
questions = [
    'How often do enterprise customers receive an update?',
    'Who may restart production services?',
    'What is the capital of France?',
]

for threshold in (0.10, 0.20, 0.50):
    decisions = [answer_with_citations(q, chunks, min_score=threshold) for q in questions]
    print(f'min_score={threshold:.2f}', [(d.abstained, d.reason) for d in decisions])


## 6. Deliberate failure: source existence is not claim support

Ask “Can support restart a service?” The corpus contains `restart`, but the actual rule says support **cannot** restart production services and that approval is required. A system that retrieves a source yet paraphrases it carelessly can cause harm.

The current deterministic baseline returns raw evidence rather than inventing a paraphrase. In a model-backed system, require claim-level evidence, preserve the exact quoted support for high-risk claims, and run factuality evaluation on a held-out set.


In [ ]:
high_risk = answer_with_citations('Can support restart a production service?', chunks)
print(render_markdown(high_risk))
print('\nAudit:', audit_answer(high_risk, chunks))


## 7. Practical policy checklist

- Retrieve only documents the current user is authorized to see.
- Carry stable citation IDs, source locations, versions, and retrieval trace IDs.
- Validate citation IDs against the retrieved evidence set before rendering.
- Make no-answer states explicit, reasoned, and measurable.
- Separate factual claims from recommendations; cite the facts and label the recommendation.
- Evaluate citation correctness, completeness, freshness, and user usefulness—not only whether links render.


## 8. Exercise: write a claim-level response contract

Extend the `CitedAnswer` model with a list of claims. Each claim should contain:

```text
claim text
supporting chunk IDs
claim type: fact | recommendation | unknown
```

Then add tests for:

1. a claim with a citation ID that was never retrieved;
2. an unsupported question that must remain an abstention;
3. a response that separates “the handbook says” from “I recommend”.

**Success criterion:** your validator rejects invalid claim provenance before a user sees the answer.


## 9. Claims make citation completeness testable

A source list at the end of a paragraph hides which source supports which factual assertion. Model a claim separately, attach its evidence IDs, then validate the graph before rendering. A claim can be a fact, recommendation, or unknown; only factual claims should be presented as supported by the corpus.

The lexical support check below is a deliberately transparent baseline. It catches an obvious mismatch but does not prove natural-language entailment. For high-risk domains, combine deterministic invariants, human-reviewed cases, and a calibrated faithfulness/entailment evaluator.


In [ ]:
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/beginner/04-citations-abstention/lab.py')).items() if not name.startswith('_')})

approval_chunk = next(chunk for chunk in chunks if chunk.chunk_id == 'harborline-support-7')
approval_citation = Citation(approval_chunk.chunk_id, approval_chunk.source, 0.92, approval_chunk.section)
claim_level_answer = CitedAnswer(
    text='Support cannot restart production services without approval.',
    citations=(approval_citation,),
    abstained=False,
    reason='grounded-evidence',
    claims=(Claim(
        'approval-boundary',
        'Support cannot restart production services without incident-command approval.',
        (approval_chunk.chunk_id,),
    ),),
)
print('known citation IDs:', claims_have_known_citations(claim_level_answer))
print('claim support:', audit_claim_support(claim_level_answer, chunks))
print(audit_answer(claim_level_answer, chunks))


## 10. Deliberate failure: a real source that does not support the claim

Citation validity and claim support are different. The following answer uses a real retrieved ID but makes an unsupported promise. The deterministic audit should flag it. This failure is common in model-backed systems: the citation exists and looks credible, but the answer stretches beyond the passage.


In [ ]:
unsafe_claim = CitedAnswer(
    text='Support can restart production services immediately.',
    citations=(approval_citation,),
    abstained=False,
    reason='grounded-evidence',
    claims=(Claim(
        'unsafe-restart',
        'Support can restart production services immediately.',
        (approval_chunk.chunk_id,),
    ),),
)
print('citation ID is known:', claims_have_known_citations(unsafe_claim))
print('lexical support / unsupported:', audit_claim_support(unsafe_claim, chunks))
print('full audit:', audit_answer(unsafe_claim, chunks))


## 11. Authorization belongs before retrieval

A relevant document may still be unavailable to the caller. The starter supports an allow-list so we can observe the boundary deterministically. In production, bind a tenant/ACL filter to authenticated caller claims in the retrieval backend, preserve the decision in the trace, and ensure logs and caches follow the same boundary.


In [ ]:
visible = {'harborline-policy-1', 'harborline-policy-2', 'harborline-policy-3'}
restricted = answer_with_citations(
    'Who may restart production services?',
    chunks,
    allowed_chunk_ids=visible,
)
print(restricted.abstained, restricted.reason)
print(render_markdown(restricted))
assert all(citation.chunk_id in visible for citation in restricted.citations)


## 12. Turn abstention into a user-safe workflow

A no-answer should state the decision reason and offer the next safe action. The action is a product and risk decision, not an LLM improvisation. For Harborline, an unsupported policy question can ask for a canonical document; an ambiguous operational question can route to an incident commander; a restricted question can explain that access is required without exposing the hidden source.

```text
insufficient evidence -> state corpus gap -> ask for source / escalate
ambiguous evidence    -> state uncertainty -> clarify / route to owner
restricted evidence   -> state access limit -> request authorized path
stale evidence        -> state freshness issue -> re-verify canonical source
```


## 13. Evaluation matrix

Test each layer separately. Retrieval metrics ask whether the evidence arrived; citation metrics ask whether identifiers are valid and complete; faithfulness asks whether wording stays within evidence; abstention accuracy asks whether the terminal decision is useful. Ragas offers context precision and faithfulness metrics, but a model judge is another component to validate and monitor.

| Case | Expected decision | Expected audit outcome |
| --- | --- | --- |
| supported policy | answer | known claim citation + support |
| France question | abstain | no citations |
| small rank margin | abstain | `ambiguous-evidence` |
| restricted policy | abstain or alternate answer | no hidden citation |
| invented citation ID | block | claim ID not known |
| real but irrelevant citation | block/revise | unsupported claim |


## 14. Capstone and checkpoint

Build a Harborline policy release gate. Given a question, caller scope, corpus, and retrieval policy, show an answer only if every claim has visible evidence and the audit passes. Otherwise return a reason code, what was searched, and the safe next step.

1. Why is a retrieved citation not sufficient proof for a claim?
2. Why must authorization occur before retrieval rather than after answer generation?
3. Which cases should be in a responsible abstention evaluation set?
4. What can lexical claim-support catch, and what requires a stronger evaluation?
5. What trace fields make a disputed answer diagnosable after deployment?

### References

- [RAG evaluation guide in this repository](../../docs/evaluation.md)
- NIST, [Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)
- [Ragas available metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/)
- [OWASP Top 10 for LLM Applications](https://genai.owasp.org/llm-top-10/)
